In [ ]:
import pandas as pd
import numpy as np

# ------------ Load the data from raw imports ------------ 
crsp = pd.read_csv('wrds_data.csv') 
hmi = pd.read_excel('hmi.xls', header=None) 
treasury = pd.read_csv('10_year_treasury.csv') 
# ------------  Data Cleaning and Aggregation ------------ 
data_start_idx = hmi[hmi[0] == 1985].index[0]
hmi = hmi.iloc[data_start_idx:].reset_index(drop=True)
hmi.columns = ['Year', 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
hmi['Year'] = hmi['Year'].astype(int)
hmi = hmi.melt(id_vars=['Year'], var_name='Month', value_name='HMI') # row per year -> row per month per year
hmi = hmi.sort_values(['Year', 'Month']).reset_index(drop=True)
hmi['HMI_lag1'] = hmi['HMI'].shift(1)
hmi = hmi.dropna(subset=['HMI_lag1']).reset_index(drop=True)

treasury['date'] = pd.to_datetime(treasury['observation_date'], format='%Y-%m-%d')
treasury['Month'] = treasury['date'].dt.month
treasury['Year'] = treasury['date'].dt.year
treasury = treasury.groupby(['Year', 'Month'])['DGS10'].mean().reset_index() # from daily to monthly

crsp['date'] = pd.to_datetime(crsp['date'])
crsp['Month'] = crsp['date'].dt.month
crsp['Year'] = crsp['date'].dt.year
crsp['RET'] = pd.to_numeric(crsp['RET'], errors='coerce')
crsp = crsp.dropna(subset=['RET']).copy()
crsp['illiq_daily'] = crsp['RET'].abs() / (crsp['VOL'])
def calculate_monthly_ret(series):
    return (series + 1).prod() - 1
df_monthly = crsp.groupby(['TICKER', 'Year', 'Month']).agg({
    'illiq_daily': 'mean',
    'RET': calculate_monthly_ret,    
    'VOL': 'sum'                     
}).reset_index()
df_monthly.rename(columns={'RET': 'monthly_ret', 'illiq_daily': 'illiq_avg'}, inplace=True)

# ------------ Merging and Final Clean up------------ 
df = df_monthly.merge(hmi[['Year', 'Month', 'HMI_lag1']], on=['Year', 'Month'], how='inner')
df = df.merge(treasury, on=['Year', 'Month'], how='inner')
df['log_illiq'] = np.log(df['illiq_avg'])
df = df.sort_values(['TICKER', 'Year', 'Month']).reset_index(drop=True)
treatment_tickers = ['ESS', 'AVB', 'INVH', 'UDR', 'MAA', 'CPT', 'AMH']
control_tickers = ['WELL', 'AMT', 'EQIX', 'PLD']
df['treatment'] = df['TICKER'].apply(lambda x: 1 if x in treatment_tickers else (0 if x in control_tickers else np.nan))
df.columns = df.columns.str.lower()
df
# the downloaded query in drive isn't quite the current specification // df[df['treatment'].isna()]['ticker'].unique()
# also need to double check my illiq calculation (number appears very low; read the paper again to figure out expected distribution)

,ticker,year,month,illiq_avg,monthly_ret,vol,hmi_lag1,dgs10,log_illiq,treatment
0,AMB,2010,1,1.151464e-08,-0.060664,24564200.0,16,3.733158,-18.279646,NaN
1,AMB,2010,2,9.844087e-09,0.014167,31560200.0,15,3.691053,-18.436395,NaN
2,AMB,2010,3,8.693246e-09,0.130650,33093500.0,17,3.727391,-18.560719,NaN
3,AMB,2010,4,7.223910e-09,0.022760,58414100.0,15,3.846818,-18.745869,NaN
4,AMB,2010,5,1.102104e-08,-0.069277,52424900.0,19,3.420000,-18.323460,NaN
...,...,...,...,...,...,...,...,...,...,...
1686,WELL,2024,8,3.087651e-09,0.090994,59322087.0,41,3.870909,-19.595855,0.0
1687,WELL,2024,9,3.560994e-09,0.060905,64856745.0,39,3.723500,-19.453226,0.0
1688,WELL,2024,10,4.568061e-09,0.053503,51547361.0,41,4.095455,-19.204177,0.0
1689,WELL,2024,11,3.702272e-09,0.029556,54004044.0,43,4.355789,-19.414319,0.0
